In [91]:
# --- CELL 1: Imports and Geometry Setup ---
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

# Parameters
L = 2.0
L_pml = 2.0
Lx, Ly = L/2, L/64

# Create the boxes for the domains
b_bot = Box(Pnt(0,0,-L_pml), Pnt(Lx,Ly,0))
b_bot.mat("pml_bot")

b_phys = Box(Pnt(0,0,0), Pnt(Lx,Ly,L))
b_phys.mat("phys")

b_top = Box(Pnt(0,0,L), Pnt(Lx,Ly,L+L_pml))
b_top.mat("pml_top")

# Glue them together to ensure conformal mesh interfaces
shape = Glue([b_bot, b_phys, b_top])

# Tag the boundaries for Boundary Conditions
shape.faces.Max(Z).name = "top"
shape.faces.Min(Z).name = "bottom"

# Identify Periodic Faces (on the outer boundaries of the glued geometry)
shape.faces.Max(X).Identify(shape.faces.Min(X), "periodic_x")
shape.faces.Max(Y).Identify(shape.faces.Min(Y), "periodic_y")

geo = OCCGeometry(shape)



In [100]:
# --- CELL 2: Mesh Generation and FE Space ---
# Generate the mesh with an appropriate element size
h_max = 0.05
mesh = Mesh(geo.GenerateMesh(maxh=h_max))
Draw(mesh)
# Define the Finite Element Space
# We use complex=True for the frequency domain and order=2 for better wave resolution.
# "fixed" boundaries and "pml_end" (to strictly truncate waves) are set to 0.
fes = Compress(Periodic(H1(mesh, order=4, complex=True)))
print("Degrees of freedom:", fes.ndof)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Degrees of freedom: 207876


In [101]:
import math

# --- CELL 3: Formulating the Explicit PML ---
omega = 2*math.pi*300.0      # Angular frequency
c = 340.0         # Speed of sound
k = omega / c     # Wavenumber
c_real = c.real 
m = 2                               # Quadratic profile
R0 = 1e-6                           # Target reflection coefficient (0.01%)
sigma0 = ((m + 1) * c_real / (2 * L_pml)) * math.log(1 / R0)
print(f"Calculated sigma0 for PML: {sigma0:.2f}")

# Incident angles (e.g., 45 degrees)
theta = (math.pi / 40) * 0.0  # Polar angle (0 for normal incidence)
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

# Transverse wave vector for the Floquet shift
k_vec = CF((kx, ky, 0)) 

# Coordinate functions
x, y, z = x, y, z
print(f"Coordinate functions: x={x}, y={y}, z={z}")

# Calculate distance into the PML (0 inside the physical domain)
dist_top = IfPos(z - L, z - L, 0)
dist_bot = IfPos(-z, -z, 0)
dist = dist_top + dist_bot

# Quadratic damping profile
# sigma = sigma0 * (dist / L_pml)**2
sigma = sigma0 * (2* (dist / L_pml)**2 - (dist / L_pml)**4)

# Complex stretching factor
sz = 1 + 1j * (sigma / omega)

# The anisotropic stretching matrix S = diag(sz, sz, 1/sz)
S = CF((1, 0, 0, 
        0, 1, 0, 
        0, 0, 1/sz**2), dims=(3,3))

# Gaussian Point Source at (L/8, L/8, L/2)
z0 = L/2
sigma_s = 0.2
source_expr = exp( -((z-z0)**2 + (x-Lx/8)**2 + (y-Ly/2)**2) / (2*sigma_s**2) )

# Setup weak forms
u, v = fes.TnT()
grad_u = grad(u) + 1j * k_vec * u
grad_v = grad(v) - 1j * k_vec * v 
a = BilinearForm(fes)
a += S * grad_u * grad_v * sz* dx - (k**2) * sz * u * v * dx
# a += grad(u) * grad(v) * dx - (k**2) * u * v * dx

f = LinearForm(fes)
# Apply source only in the physical domain
f += source_expr * v * sz*dx


gfu = GridFunction(fes)
with TaskManager():
    a.Assemble()
    f.Assemble()
    gfu.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec
    
print("Solve complete!")

Calculated sigma0 for PML: 3522.96
Coordinate functions: x=coef coordinate x, real
, y=coef coordinate y, real
, z=coef coordinate z, real

Solve complete!


In [102]:
# --- CELL 4: Plot ---

phase = exp(1j * (kx * x + ky * y))
p_true = gfu * phase
Draw(p_true, mesh, "Acoustic_Pressure_Magnitude", animate_complex=True)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene